In [4]:
%matplotlib notebook
from scipy.ndimage import sobel, laplace
from scipy.stats import wilcoxon
import numpy as np
import pickle
import matplotlib.pyplot as plt


def rescale_values(image,max_val,min_val):
    '''
    image - numpy array
    max_val/min_val - float
    '''
    return (image-image.min())/(image.max()-image.min())*(max_val-min_val)+min_val

SEED=1234
np.random.seed(SEED)
# torch.manual_seed(SEED)

In [5]:
# Scale matrix to sum to 1
def sum_to_1(mat):
    return mat / np.sum(mat)

def rand_baseline(data: np.array):
    rands = np.random.uniform(low=-1.0, high=1.0, size=(data.shape))
    return sum_to_1(rands)

def x_baseline(data: np.array):
    rgb_weights = [0.2989, 0.5870, 0.1140]
    greyscale = np.dot(data[...,:3], rgb_weights)
    return sum_to_1(greyscale)

In [3]:
# # Confounder Data
# with open('confounder_train128.pkl', 'rb') as f:
#     confounder_train = pickle.load(f)
#     confounder_train = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in confounder_train]

# with open('confounder_val128.pkl', 'rb') as f:
#     confounder_val = pickle.load(f)
#     confounder_val = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in confounder_val]

# with open('confounder_test128.pkl', 'rb') as f:
#     confounder_test = pickle.load(f)
#     confounder_test = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in confounder_test]

    
# # Suppressor Data
# with open('suppressor_train128.pkl', 'rb') as f:
#     supressor_train = pickle.load(f)
#     supressor_train = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in supressor_train]

# with open('suppressor_validation128.pkl', 'rb') as f:
#     supressor_val = pickle.load(f)
#     supressor_val = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in supressor_val]

# with open('suppressor_test128.pkl', 'rb') as f:
#     supressor_test = pickle.load(f)
#     supressor_test = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in supressor_test]


# # No Watermark Data
# with open('no_mark_train128.pkl', 'rb') as f:
#     no_mark_train = pickle.load(f)
#     no_mark_train = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in no_mark_train]

# with open('no_mark_validation128.pkl', 'rb') as f:
#     no_mark_val = pickle.load(f)
#     no_mark_val = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in no_mark_val]

# with open('no_mark_test128.pkl', 'rb') as f:
#     no_mark_test = pickle.load(f)
#     no_mark_test = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in no_mark_test]

In [4]:
# with open('mark_all128.pkl', 'rb') as f:
#     watermark_dataset = pickle.load(f)
#     watermark_dataset = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in watermark_dataset]

# with open('no_mark_test128.pkl', 'rb') as f:
#     no_watermark_dataset = pickle.load(f)
#     no_watermark_dataset = [[rescale_values(i[0],1,0).transpose(2,0,1),i[1]] for i in no_watermark_dataset]

In [22]:
# Load and arrange all results for plotting
def get_energies(variable=False, rescaled=False):
    variable_string = ''
    rescaled_string = ''
    if variable:
        variable_string = '_variable'

    if rescaled:
        rescaled_string = '_rescaled'


    energies=[]
    methods = ['deconv', 'int_grads', 'shap', 'lrp', 'lrp_ab', 'laplace', 'sobel', 'x'] # lime just nan values
    n_test = 3000

    energy_files = ['energy_water_conf', 'energy_water_sup', 'energy_water_no', 'energy_no_water_conf', 'energy_no_water_sup', 'energy_no_water_no']


    
    for method in methods:
        comb_water_conf_cat = []
        comb_water_conf_dog = []
        comb_water_sup = []
        comb_water_no = []
        
        comb_no_water_conf_cat = []
        comb_no_water_conf_dog = []
        comb_no_water_sup = []
        comb_no_water_no = []
    
        for split in range(4):

            with open(f'./artifacts/split_{split}_confounder{variable_string}_test{rescaled_string}.pkl', 'rb') as file:
                data = pickle.load(file)
            
            labels = data[1]
            
            for model_ind in range(5):
                with open(f'./energies/energy_water_conf_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file:
                    energy_water_conf = pickle.load(file)
        
                with open(f'./energies/energy_water_sup_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file: 
                    energy_water_sup = pickle.load(file)
        
                with open(f'./energies/energy_water_no_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file:
                    energy_water_no = pickle.load(file)
        
                with open(f'./energies/energy_no_water_conf_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file:
                    energy_no_water_conf = pickle.load(file)
        
                with open(f'./energies/energy_no_water_sup_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file:
                    energy_no_water_sup = pickle.load(file)
        
                with open(f'./energies/energy_no_water_no_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file:
                    energy_no_water_no = pickle.load(file)

                
                # for energy_ind, energy in enumerate([energy_water_conf, energy_water_sup, energy_water_no, energy_no_water_conf, energy_no_water_sup, energy_no_water_no]):
                #     if np.isnan(np.asarray(energy[method][:n_test])).any():
                #         print('nan values', energy_files[energy_ind], 'split', split, 'model', model_ind,'method', method, np.where(np.isnan(np.asarray(energy[method][:n_test]))))
    
                #     if np.isinf(np.asarray(energy[method][:n_test])).any():
                #         print('inf values',  energy_files[energy_ind], 'split', split, 'model', model_ind,'method', method, np.where(np.isinf(np.asarray(energy[method][:n_test]))))

                
                
                # print(energy_water_no.keys())
    
                water_conf = energy_water_conf[method][:n_test]
                no_water_conf = energy_no_water_conf[method][:n_test]
            
                # cat labels = 0, dog = 1; 20% cat with WM, 80% dog with WM
                water_conf_cat = np.asarray(water_conf)[np.where(labels == 0)[0]] 
                no_water_conf_cat = np.asarray(no_water_conf)[np.where(labels == 0)[0]] 
                
                water_conf_dog = np.asarray(water_conf)[np.where(labels == 1)[0]] 
                no_water_conf_dog = np.asarray(no_water_conf)[np.where(labels == 1)[0]] 
                
                comb_water_conf_cat.extend(water_conf_cat)
                comb_water_conf_dog.extend(water_conf_dog)
                
                comb_water_sup.extend(energy_water_sup[method][:n_test])
        
                if method == 'laplace' or method == 'sobel' or method == 'x':
                    comb_water_no.extend(energy_water_sup[method][:n_test])
                    comb_no_water_conf_cat.extend(np.asarray(energy_no_water_sup[method][:n_test])[np.where(labels == 0)[0]] )
                    comb_no_water_conf_dog.extend(np.asarray(energy_no_water_sup[method][:n_test])[np.where(labels == 1)[0]] )
                else:
                    comb_water_no.extend(energy_water_no[method][:n_test])
                    comb_no_water_conf_cat.extend(no_water_conf_cat)
                    comb_no_water_conf_dog.extend(no_water_conf_dog)
                    
        
                # comb_no_water_conf.extend(energy_no_water_conf[method][:n_test])
                comb_no_water_sup.extend(energy_no_water_sup[method][:n_test])
                comb_no_water_no.extend(energy_no_water_no[method][:n_test])
                
                
        energies.append([[comb_no_water_conf_cat, comb_water_conf_cat],
                    [comb_no_water_conf_dog, comb_water_conf_dog],
                    [comb_no_water_sup, comb_water_sup],
                    [comb_no_water_no, comb_water_no]])

        
    return energies

In [24]:
import os
from PIL import Image

def rescale_values(image,max_val,min_val):
    '''
    image - numpy array
    max_val/min_val - float
    '''
    return (image-image.min())/(image.max()-image.min())*(max_val-min_val)+min_val

folder=os.getcwd()
watermark_path=folder+'/watermark banner.jpg'
watermark = Image.open(watermark_path)
w=128
h=int(watermark.size[1]*w/watermark.size[0])
watermark = watermark.resize((w,h))
watermark=np.array(watermark)
rgb=rescale_values(watermark,1,0)
r, g, blue = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
gray = 1-(0.2989 * r + 0.5870 * g + 0.1140 * blue)
white=np.ones((w,w))
white[0:gray.shape[0],0:gray.shape[1]]=gray


fixed_mask = 1.0 * (white<1)
rgb_weights = [0.2989, 0.5870, 0.1140]

def energy(att, watermark):
    
    
    image_size=att.shape[0]*att.shape[1]
    watermark_size=np.sum(watermark)
    
    watermark_att=att*watermark
    watermark_energy=np.sum(watermark_att)
    
    image_energy=np.sum(att)

    
    # Gives energy(watermark) = 7.858
    energy=(watermark_energy/watermark_size)/(image_energy/image_size)
    
    return energy

def get_energies_x(fixed_mask, variable=False, rescaled=False):
    variable_string = ''
    rescaled_string = ''
    if variable:
        variable_string = '_variable'

    if rescaled:
        rescaled_string = '_rescaled'

    energies_wm = []
    energies_no = []

    energies_wm_cat = []
    energies_wm_dog = []

    energies_no_cat = []
    energies_no_dog = []
    
    for split in range(4):
        with open(f'./artifacts/split_{split}_all_watermark{variable_string}_test{rescaled_string}.pkl', 'rb') as file:
            data_wm = pickle.load(file)

        with open(f'./artifacts/split_{split}_no_watermark{variable_string}_test{rescaled_string}.pkl', 'rb') as file:
            data_no = pickle.load(file)
        
        for i in range(data_wm[0].shape[0]):
            watermark = fixed_mask
            if variable:
                watermark = data_wm[-1][i]

            energies_wm.append(energy(np.dot(np.abs(data_wm[0][i])[...,:3], rgb_weights), watermark))
            energies_no.append(energy(np.dot(np.abs(data_no[0][i])[...,:3], rgb_weights), watermark))

        energies_wm_cat.extend(np.asarray(energies_wm)[np.where(data_wm[1] == 0)[0]]) 
        energies_wm_dog.extend(np.asarray(energies_wm)[np.where(data_wm[1] == 1)[0]]) 

        energies_no_cat.extend(np.asarray(energies_no)[np.where(data_no[1] == 0)[0]]) 
        energies_no_dog.extend(np.asarray(energies_no)[np.where(data_no[1] == 1)[0]]) 

        
    
    
    return energies_wm, energies_no, [energies_no_cat, energies_wm_cat], [energies_no_dog, energies_wm_dog]

## Hypothesis 1
- if the XAI methods work as expected as confounder detectors, the relative energy should differ betwen W and NO-W test images only for the confounder case, but not for the other two.
- Using a [paired Wilcoxon signed-rank test](https://en.wikipedia.org/wiki/Wilcoxon_signed-rank_test) to test significance
- Evaluate differences between E(W_i) and E(NoW_i) for all three models for each given XAI method

In [26]:
# Effect size as rank-biserial correlation
# aka (test statistic / total rank sum)

methods = ['deconv', 'int_grads', 'shap', 'lrp', 'lrp_ab', 'laplace', 'sobel', 'x']
models = ['confounder_cat', 'confounder_dog', 'suppressor', 'no watermark']


for variable in [False, True]:
    for rescaled in [False, True]:
        energies = get_energies(variable, rescaled)
        
        print('VARIABLE:', variable, 'RESCALED', rescaled)
        for i, results in enumerate(energies):
            if i != 7:
                for j, result in enumerate(results):
                    # print(methods[i], models[j], len(result[1]), len(result[0]))
                    
                    res = wilcoxon(result[1], result[0], alternative='greater')
                    rank_sum = sum(np.where(np.array(result[1]) ==  np.array(result[1]))[0])
                    
                    print(methods[i], models[j], res.statistic/rank_sum, res.pvalue)
            else:
                energies_wm, energies_no, x_cat, x_dog  = get_energies_x(fixed_mask, variable=variable, rescaled=rescaled)

                res = wilcoxon(x_cat[1], x_cat[0], alternative='greater')
                rank_sum = sum(np.where(np.array(x_cat[1]) ==  np.array(x_cat[1]))[0])
                print(methods[i], 'confounder_cat', res.statistic/rank_sum, res.pvalue)

                res = wilcoxon(x_dog[1], x_dog[0], alternative='greater')
                rank_sum = sum(np.where(np.array(x_dog[1]) ==  np.array(x_dog[1]))[0])
                print(methods[i], 'confounder_dog', res.statistic/rank_sum, res.pvalue)

                res = wilcoxon(energies_wm, energies_no, alternative='greater')
                rank_sum = sum(np.where(np.array(energies_wm) ==  np.array(energies_wm))[0])
                print(methods[i], 'suppressor', res.statistic/rank_sum, res.pvalue)

                res = wilcoxon(energies_wm, energies_no, alternative='greater')
                rank_sum = sum(np.where(np.array(energies_wm) ==  np.array(energies_wm))[0])
                print(methods[i], 'no watermark', res.statistic/rank_sum, res.pvalue)
                
        print('\n\n')

VARIABLE: False RESCALED False
deconv confounder_cat 1.0000030125130408 0.0
deconv confounder_dog 1.000077158607577 0.0
deconv suppressor 0.9986210712643252 0.0
deconv no watermark 1.0000228123620718 0.0
int_grads confounder_cat 1.0001111172842936 0.0
int_grads confounder_dog 1.0001111049379285 0.0
int_grads suppressor 1.0000554737631602 0.0
int_grads no watermark 1.0000555354932699 0.0
shap confounder_cat 1.0001098147427943 0.0
shap confounder_dog 1.000109740664605 0.0
shap suppressor 1.0000450814991775 0.0
shap no watermark 1.000052935112395 0.0
lrp confounder_cat 1.0001110493792864 0.0
lrp confounder_dog 1.0001111111111112 0.0
lrp suppressor 1.0000555216039952 0.0
lrp no watermark 1.0000554969119513 0.0
lrp_ab confounder_cat 1.000104289744492 0.0
lrp_ab confounder_dog 1.0001039255267268 0.0
lrp_ab suppressor 0.999426368325046 0.0
lrp_ab no watermark 0.9998258717063129 0.0
laplace confounder_cat 0.9960700903136595 0.0
laplace confounder_dog 0.9824293633596928 0.0
laplace suppressor 0

In [17]:
len(energies[5][0][0]), len(energies[5][0][1])

(36000, 18000)

In [13]:
len(energies)

8

In [29]:
test_result = energies['deconv']['conf']

res = wilcoxon(test_result[1], test_result[0], alternative='greater')
print(res)

WilcoxonResult(statistic=101814272.0, pvalue=0.0)


In [29]:
split = 0
variable_string = ''
rescaled_string = ''
with open(f'./artifacts/split_{split}_confounder{variable_string}_test{rescaled_string}.pkl', 'rb') as file:
    data_conf = pickle.load(file)

with open(f'./artifacts/split_{split}_all_watermark{variable_string}_test{rescaled_string}.pkl', 'rb') as file:
    data_wm = pickle.load(file)

labels_conf = data_conf[1]
labels_wm = data_wm[1]

In [11]:
# lapl_no, lapl_wm = lapl
# sob_no, sob_wm = sob

## Hypothesis 2
if the XAI methods work as expected as confounder detectors, then the relative energy on W test images should be higher for confounded training data then for the other two. If this is the case - how big is the effect?
- Meaning, how much more watermark energy is attributed by models trained on confounded data compared to models trained on unconfounded data.
- Calculate E(Wˆtr=Conf_i) - E(Wˆtr=Supp_i) and E(Wˆtr=Conf_i) - E(Wˆtr=NoW_i) for all XAI methods

In [35]:
# Separated cats and dogs energies required for hypothesis testing of SUP and NO
def get_energies_separated(variable=False, rescaled=False):
    variable_string = ''
    rescaled_string = ''
    if variable:
        variable_string = '_variable'

    if rescaled:
        rescaled_string = '_rescaled'


    energies=[]
    methods = ['deconv', 'int_grads', 'shap', 'lrp', 'lrp_ab'] # lime just nan values
    n_test = 3000

    energy_files = ['energy_water_conf', 'energy_water_sup', 'energy_water_no', 'energy_no_water_conf', 'energy_no_water_sup', 'energy_no_water_no']


    
    for method in methods:
        comb_water_conf_cat = []
        comb_water_conf_dog = []
        comb_water_sup_cat = []
        comb_water_sup_dog = []
        comb_water_no_cat = []
        comb_water_no_dog = []
        
        comb_no_water_conf_cat = []
        comb_no_water_conf_dog = []
        comb_no_water_sup_cat = []
        comb_no_water_sup_dog = []
        comb_no_water_no_cat = []
        comb_no_water_no_dog = []
    
        for split in range(4):

            with open(f'./artifacts/split_{split}_all_watermark{variable_string}_test{rescaled_string}.pkl', 'rb') as file:
                data = pickle.load(file)
            
            labels = data[1]
            
            for model_ind in range(5):
                with open(f'./energies/energy_water_conf_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file:
                    energy_water_conf = pickle.load(file)
        
                with open(f'./energies/energy_water_sup_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file: 
                    energy_water_sup = pickle.load(file)
        
                with open(f'./energies/energy_water_no_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file:
                    energy_water_no = pickle.load(file)
        
                with open(f'./energies/energy_no_water_conf_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file:
                    energy_no_water_conf = pickle.load(file)
        
                with open(f'./energies/energy_no_water_sup_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file:
                    energy_no_water_sup = pickle.load(file)
        
                with open(f'./energies/energy_no_water_no_pred{variable_string}{rescaled_string}_{split}_{model_ind}.pickle', 'rb') as file:
                    energy_no_water_no = pickle.load(file)

                
                # for energy_ind, energy in enumerate([energy_water_conf, energy_water_sup, energy_water_no, energy_no_water_conf, energy_no_water_sup, energy_no_water_no]):
                #     if np.isnan(np.asarray(energy[method][:n_test])).any():
                #         print('nan values', energy_files[energy_ind], 'split', split, 'model', model_ind,'method', method, np.where(np.isnan(np.asarray(energy[method][:n_test]))))
    
                #     if np.isinf(np.asarray(energy[method][:n_test])).any():
                #         print('inf values',  energy_files[energy_ind], 'split', split, 'model', model_ind,'method', method, np.where(np.isinf(np.asarray(energy[method][:n_test]))))

                
                
                # print(energy_water_no.keys())
    
                water_conf = energy_water_conf[method][:n_test]
                no_water_conf = energy_no_water_conf[method][:n_test]
            
                # cat labels = 0, dog = 1; 20% cat with WM, 80% dog with WM
                water_conf_cat = np.asarray(water_conf)[np.where(labels == 0)[0]] 
                no_water_conf_cat = np.asarray(no_water_conf)[np.where(labels == 0)[0]] 
                water_conf_dog = np.asarray(water_conf)[np.where(labels == 1)[0]] 
                no_water_conf_dog = np.asarray(no_water_conf)[np.where(labels == 1)[0]] 

                water_sup_cat = np.asarray(energy_water_sup[method][:n_test])[np.where(labels == 0)[0]] 
                no_water_sup_cat = np.asarray(energy_no_water_sup[method][:n_test])[np.where(labels == 0)[0]] 
                water_sup_dog = np.asarray(energy_water_sup[method][:n_test])[np.where(labels == 1)[0]] 
                no_water_sup_dog = np.asarray(energy_no_water_sup[method][:n_test])[np.where(labels == 1)[0]] 

                water_no_cat = np.asarray(energy_water_no[method][:n_test])[np.where(labels == 0)[0]] 
                no_water_no_cat = np.asarray(energy_no_water_no[method][:n_test])[np.where(labels == 0)[0]] 
                water_no_dog = np.asarray(energy_water_no[method][:n_test])[np.where(labels == 1)[0]] 
                no_water_no_dog = np.asarray(energy_no_water_no[method][:n_test])[np.where(labels == 1)[0]] 
                
                
                comb_water_conf_cat.extend(water_conf_cat)
                comb_water_conf_dog.extend(water_conf_dog)

                comb_water_sup_cat.extend(water_sup_cat)
                comb_water_sup_dog.extend(water_sup_dog)

                comb_water_no_cat.extend(water_no_cat)
                comb_water_no_dog.extend(water_no_dog)

                
                comb_no_water_conf_cat.extend(no_water_conf_cat)
                comb_no_water_conf_dog.extend(no_water_conf_dog)

                comb_no_water_sup_cat.extend(no_water_sup_cat)
                comb_no_water_sup_dog.extend(no_water_sup_dog)

                comb_no_water_no_cat.extend(no_water_no_cat)
                comb_no_water_no_dog.extend(no_water_no_dog)

                
                
                
        energies.append([[comb_no_water_conf_cat, comb_water_conf_cat],
                    [comb_no_water_conf_dog, comb_water_conf_dog],
                    [comb_no_water_sup_cat, comb_water_sup_cat],
                    [comb_no_water_sup_dog, comb_water_sup_dog],
                    [comb_no_water_no_cat, comb_water_no_cat],
                    [comb_no_water_no_dog, comb_water_no_dog]])

        
    return energies

In [37]:
for variable in [False,True]:
    for rescaled in [False,True]:
        energies = get_energies_separated(variable, rescaled)
        
        print('VARIABLE:', variable, 'RESCALED', rescaled)
        for i, results in enumerate(energies):
            # try
            res_sup_cat = wilcoxon(results[0][1], results[2][1], alternative='greater')
            res_sup_dog = wilcoxon(results[1][1], results[3][1], alternative='greater')
            
            #  E(Wˆtr=Conf_i) - E(Wˆtr=NoW_i) 
            # res_no = wilcoxon(results['conf'][1], results['no'][1], alternative='greater')

            res_no_cat = wilcoxon(results[0][1], results[4][1], alternative='greater')
            res_no_dog = wilcoxon(results[1][1], results[5][1], alternative='greater')
            
            rank_sum = sum(np.where(np.array(results[0][1]) ==  np.array(results[0][1]))[0])
    
            print(methods[i], 'CONF_CAT - SUPP_CAT', res_sup_cat.statistic/rank_sum, res_sup_cat.pvalue)
            print(methods[i], 'CONF_DOG - SUPP_DOG', res_sup_dog.statistic/rank_sum, res_sup_dog.pvalue)
            print(methods[i], 'CONF_CAT - NO_CAT', res_no_cat.statistic/rank_sum, res_no_cat.pvalue)
            print(methods[i], 'CONF_DOG - NO_DOG', res_no_dog.statistic/rank_sum, res_no_dog.pvalue)
            
            # except:
            #     print(method, 'CONF - SUPP')
            #     print(methods[i], 'CONF - NO')

                
        print('\n\n')



# for method, results in energies.items():
#     #  E(Wˆtr=Conf_i) - E(Wˆtr=Supp_i) 
#     try:
#         res_sup = wilcoxon(results['conf'][1], results['sup'][1], alternative='greater')
        
#         #  E(Wˆtr=Conf_i) - E(Wˆtr=NoW_i) 
#         res_no = wilcoxon(results['conf'][1], results['no'][1], alternative='greater')

#         res_no_2 = wilcoxon(results['no'][1], results['conf'][1], alternative='greater')
        
#         rank_sum = sum(np.where(np.array(results['conf'][1]) ==  np.array(results['conf'][1]))[0])

#         print(method, 'CONF - SUPP', res_sup.statistic/rank_sum, res_sup.pvalue)
#         print(method, 'CONF - NO', res_no.statistic/rank_sum, res_no.pvalue)
#         print(method, 'NO - CONF', res_no_2.statistic/rank_sum, res_no_2.pvalue)
#     except:
#         print(method, 'CONF - SUPP')
#         print(method, 'CONF - NO')

VARIABLE: False RESCALED False
deconv CONF_CAT - SUPP_CAT 0.6810489224710015 0.0
deconv CONF_DOG - SUPP_DOG 0.9683074676988228 0.0
deconv CONF_CAT - NO_CAT 0.00020500521633917934 1.0
deconv CONF_DOG - NO_DOG 0.004403590322919175 1.0
int_grads CONF_CAT - SUPP_CAT 0.9720652999240699 0.0
int_grads CONF_DOG - SUPP_DOG 0.9698303300800661 0.0
int_grads CONF_CAT - NO_CAT 4.60766338870678e-05 1.0
int_grads CONF_DOG - NO_DOG 2.2630886901124138e-05 1.0
shap CONF_CAT - SUPP_CAT 0.9035094295362088 0.0
shap CONF_DOG - SUPP_DOG 0.9012107401028452 0.0
shap CONF_CAT - NO_CAT 0.0033042020853010353 1.0
shap CONF_DOG - NO_DOG 0.002299263539332432 1.0
lrp CONF_CAT - SUPP_CAT 0.9128739065750566 0.0
lrp CONF_DOG - SUPP_DOG 0.903061805902797 0.0
lrp CONF_CAT - NO_CAT 0.014989129025686613 1.0
lrp CONF_DOG - NO_DOG 0.009367656227815126 1.0
lrp_ab CONF_CAT - SUPP_CAT 0.660491854485743 1.8370909543425637e-304
lrp_ab CONF_DOG - SUPP_DOG 0.8745279243908612 0.0
lrp_ab CONF_CAT - NO_CAT 0.3239898821539468 1.0
lrp_ab

## Coloured MNIST

In [13]:
# method = 'shap'

# conf_conf = [[],[],[]]
# conf_sup = [[],[],[]]
# conf_no = [[],[],[]]

# sup_conf = [[],[],[]]
# sup_sup = [[],[],[]]
# sup_no = [[],[],[]]

# no_conf = [[],[],[]]
# no_sup = [[],[],[]]
# no_no = [[],[],[]]

# for model_ind in range(5):
#     file = open(f'energies_mnist_{model_ind}.pickle', 'rb')
#     energies = pickle.load(file)
    
#     for i in range(3):
#         conf_conf[i].extend(energies['conf']['conf'][i][method])
#         conf_sup[i].extend(energies['conf']['sup'][i][method])
#         conf_no[i].extend(energies['conf']['no_col'][i][method])
        
#         sup_conf[i].extend(energies['sup']['conf'][i][method])
#         sup_sup[i].extend(energies['sup']['sup'][i][method])
#         sup_no[i].extend(energies['sup']['no_col'][i][method])

#         no_conf[i].extend(energies['no_col']['conf'][i][method])
#         no_sup[i].extend(energies['no_col']['sup'][i][method])
#         no_no[i].extend(energies['no_col']['no_col'][i][method])
        
        
# combined_energies = {
#     'conf': { # conf model on conf/sup/no data
#         'conf': conf_conf,
#         'sup': conf_sup, 
#         'no_col': conf_no},
            
#     'sup': { # sup model on conf/sup/no data
#         'conf': sup_conf,
#         'sup': sup_sup, 
#         'no_col': sup_no},

#     'no_col': { # no_col model on conf/sup/no data
#         'conf': no_conf,
#         'sup': no_sup, 
#         'no_col': no_no},
# }

In [14]:
# wilcoxon(x=energies['lrp']['conf'][1], y=energies['lrp']['conf'][0], alternative='greater')